# 04 — Candidate comparison on a fresh held-out In-the-Wild subset

**Baseline (failed):** third-party Dhwani ONNX (`ee01871`), `train_peak`, as in run 20261024T123319Z (In-the-Wild pilot: AUC 0.456, EER 0.561).
**Candidate:** [AntiDeepfake Wav2Vec2-Small](https://huggingface.co/nii-yamagishilab/wav2vec-small-anti-deepfake) (NII Yamagishi Lab, 95 M params,
revision `9a13264`, **CC BY-NC-SA 4.0: research use; a commercial app would need permission**). Published In-the-Wild: EER 4.24 %, AUC 0.993
(self-reported; 5.31 % in the independent lab260 arena). In-the-Wild is **not** in its training table; VoxCeleb2 (celebrity YouTube speech) is,
so speaker overlap with In-the-Wild celebrities cannot be excluded.

**Held-out design (fixed before any score is seen)**
- In-the-Wild only (ASVspoof 2019 LA access from Colab is blocked with HTTP 403, and it is in the candidate's training set anyway).
- New seed, `HELDOUT_N` genuine + `HELDOUT_N` spoof, speaker round-robin, metadata only, **excluding the 200 pilot clip IDs** (development evidence).
  Speaker-disjointness is impossible: the pilot touched all 54 In-the-Wild speakers. Stated with every result.
- Both models score the same clips. Same 2–10 s policy (reject < 2 s, analyse first 10 s), same decoding (`dhwani_infer`), batch size 1.
- Dhwani: `train_peak`, mean of 4 s windows (unchanged). Candidate: published recipe (layer-norm, whole clip, fake = softmax index 0);
  a 4 s-window mean is recorded as a secondary number.
- Thresholds are **not** tuned: 0.5 for both (symmetric default). The card's "EER @ 0.8128" is **not** used as an operating point:
  it is the EER threshold the authors found *on In-the-Wild itself* with their whole-file pipeline, so it is tuned on our test data and
  depends on a pipeline we have not proven identical. It is shown as context only. Primary comparison is threshold-free (ROC-AUC, EER)
  with a paired bootstrap on the shared clips.
- **Conversion:** the candidate is loaded without fairseq (strict key mapping into `transformers`; architecture fields checked against
  fairseq defaults; see the docstring of `antideepfake_hf.py` for the side-by-side read of activations, norms, padding, masks, pooling and
  score direction). **Parity with the authors' code is "unverified" until cell 13 reports it.** Cell 13 builds a separate Python 3.8 venv
  with fairseq 0.12.2 and scores identical inputs with the authors' reference code.
- **Development-evidence gate (cell 9):** the candidate is first scored on the 200 pilot clips. AUC < 0.80 means **stop and investigate**;
  it does not by itself prove the mapping is wrong, and a high AUC does not prove parity. Nothing is tuned on it.
- Failures are kept; coverage is reported separately. Results are a **held-out pilot subset**, not a full-dataset benchmark.

Runtime: **GPU (T4)** recommended; the candidate runs on CUDA, Dhwani on CPU ORT as before (≈ 2 s per window).

In [ ]:
# 1. Settings (fixed before any result is seen)
HELDOUT_SEED = 20261026        # new seed (pilot used 20261025)
HELDOUT_N = 150                # per class; 300 clips total
MIN_PILOT_DEV_AUC = 0.80       # dev-evidence gate: below -> stop and investigate (not proof of a mapping error; not tuning)
CAND_WINDOW, CAND_HOP = 64000, 32000   # secondary 4 s-window variant for the candidate
EXPORT_ONNX = True             # deployment feasibility check: export candidate to ONNX + ORT parity on a few clips
TRY_FAIRSEQ_PARITY = True      # build a Python 3.8 venv (uv) with fairseq 0.12.2 and compare logits on identical inputs
PARITY_N_CLIPS = 12            # clips compared in the parity check (whole-clip inputs)
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/vcd_cache"
WORK = "/content/vcd_eval"
RUN_NAME = None

In [ ]:
# 2. Dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnxruntime", "librosa", "soundfile", "huggingface_hub",
                "transformers", "safetensors", "onnx", "uv"], check=True)

In [ ]:
%%writefile dhwani_infer.py
"""Inference wrapper for the third-party Dhwani ONNX checkpoint.

Model: ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model (MIT),
file best_model.onnx at HF revision MODEL_REVISION. We did NOT train it.

Preprocessing is reproduced from the upstream training notebook
(github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API @ efd512c,
training/training_notebook.ipynb cell 4), which is the notebook whose cell 14
exports best_model.onnx with a (1, 64000) dummy input. See
docs/UPSTREAM_AUDIT.md for why we do not use the model card's z-score recipe
as the default.

Scores are softmax outputs of an UNCALIBRATED classifier. They are model
scores, not probabilities that a clip is synthetic.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np

# ---- pinned provenance -----------------------------------------------------
MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"
MODEL_REVISION = "ee0187142973e1f02050e47d2a31c55ee0263065"
MODEL_SHA256 = "d1c232bf4d7990526804b375c15804c0ee4e9b566478b908c5a57431491b7342"
MODEL_BYTES = 1262760439

# ---- preprocessing / windowing policy ---------------------------------------
SR = 16000
WIN = 64000            # 4.0 s: training crop length and ONNX export shape
HOP = 32000            # 2.0 s: same hop as the upstream API's sliding window
MIN_DUR_S = 2.0        # brief: 2-10 s clips
MAX_DUR_S = 10.0       # longer clips: analyse the first 10 s, flagged
PEAK_TARGET_DBFS = -3.0

# ---- rejection gates (input validation only; never fed to the model) --------
SILENT_PEAK_DBFS = -50.0   # whole clip quieter than this -> silent
ACTIVE_FRAME_S = 0.025
ACTIVE_REL_DB = 35.0       # frame counts as active if within 35 dB of loudest frame
ACTIVE_ABS_DBFS = -55.0    # ...and above this absolute level
MIN_ACTIVE_S = 1.0

# Class order from upstream dataset.py: labels = [0]*real + [1]*fake.
# Documented in source; must still be confirmed empirically on known clips.
LABELS = {0: "genuine", 1: "synthetic"}
SYNTHETIC_INDEX = 1
DECISION_THRESHOLD = 0.5   # placeholder, NOT tuned on any validation set

# Controlled comparison (docs/UPSTREAM_AUDIT.md):
#   train_peak   - training recipe: peak -3 dBFS, 4 s windows             (A)
#   train_zscore - identical windows, z-score instead of peak norm        (A vs B isolates normalisation)
#   api_replica  - deployed API: first 3 s, z-score, zero-pad to 4 s     (B vs C isolates crop/padding)
PREPROCESSING_MODES = ("train_peak", "train_zscore", "api_replica")


class AudioRejected(Exception):
    """Input cannot be analysed (unreadable, empty, silent, too short)."""


def sha256_file(path: str, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _dbfs(x: float) -> float:
    return float(20 * np.log10(max(x, 1e-12)))


# ---- loading ------------------------------------------------------------------
def load_audio(path: str):
    """Return (samples [channels, n] float32 at native rate, native_sr).

    Tries soundfile first (wav/flac/ogg), then librosa/audioread (mp3, m4a, opus
    via ffmpeg). Never substitutes zeros on failure.
    """
    import soundfile as sf

    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)  # (n, ch)
        y = y.T
    except Exception as sf_err:
        try:
            import warnings

            import librosa

            with warnings.catch_warnings():  # expected audioread fallback for m4a/aac
                warnings.simplefilter("ignore", (UserWarning, FutureWarning))
                y, sr = librosa.load(path, sr=None, mono=False)
            y = np.atleast_2d(y).astype(np.float32)
        except Exception as lr_err:
            raise AudioRejected(
                f"unreadable audio (soundfile: {type(sf_err).__name__}; "
                f"librosa: {type(lr_err).__name__}: {lr_err})"
            ) from lr_err
    if y.size == 0 or y.shape[-1] == 0:
        raise AudioRejected("empty audio (0 samples)")
    if not np.all(np.isfinite(y)):
        raise AudioRejected("audio contains NaN/Inf samples")
    return y, int(sr)


def to_mono_16k(y: np.ndarray, sr: int) -> np.ndarray:
    mono = y.mean(axis=0) if y.shape[0] > 1 else y[0]
    if sr != SR:
        import librosa

        mono = librosa.resample(mono, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return mono.astype(np.float32)


def signal_stats(x: np.ndarray) -> dict:
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) if x.size else 0.0
    flen = int(ACTIVE_FRAME_S * SR)
    nfr = len(x) // flen
    if nfr:
        fr = x[: nfr * flen].reshape(nfr, flen).astype(np.float64)
        fdb = 10 * np.log10(np.mean(fr**2, axis=1) + 1e-20)
        active = (fdb > fdb.max() - ACTIVE_REL_DB) & (fdb > ACTIVE_ABS_DBFS)
        active_s = float(active.sum() * ACTIVE_FRAME_S)
    else:
        active_s = 0.0
    return {
        "peak_dbfs": round(_dbfs(peak), 2),
        "rms_dbfs": round(_dbfs(rms), 2),
        "active_s": round(active_s, 3),
        "clipping_frac": round(float(np.mean(np.abs(x) >= 0.999)) if x.size else 0.0, 5),
    }


def validate(x16: np.ndarray) -> tuple[np.ndarray, dict]:
    """Apply the 2-10 s policy and silence gates. Returns (analysed audio, info)."""
    dur = len(x16) / SR
    info = {"duration_s": round(dur, 3), "truncated": False}
    if dur < MIN_DUR_S:
        raise AudioRejected(f"too short: {dur:.2f} s < {MIN_DUR_S} s minimum")
    if dur > MAX_DUR_S:
        x16 = x16[: int(MAX_DUR_S * SR)]
        info["truncated"] = True
    info["analysed_duration_s"] = round(len(x16) / SR, 3)
    st = signal_stats(x16)
    info.update(st)
    if st["peak_dbfs"] < SILENT_PEAK_DBFS:
        raise AudioRejected(f"silent: peak {st['peak_dbfs']} dBFS < {SILENT_PEAK_DBFS} dBFS")
    if st["active_s"] < MIN_ACTIVE_S:
        raise AudioRejected(
            f"mostly silent: {st['active_s']:.2f} s active signal < {MIN_ACTIVE_S} s"
        )
    return x16, info


# ---- preprocessing --------------------------------------------------------------
def window_starts(n: int) -> list[int]:
    """4 s windows, 2 s hop, last window aligned to the clip end (full coverage)."""
    if n <= WIN:
        return [0]
    starts = list(range(0, n - WIN + 1, HOP))
    if starts[-1] + WIN < n:
        starts.append(n - WIN)
    return starts


def make_windows(x16: np.ndarray, mode: str) -> tuple[np.ndarray, list[int]]:
    """Return (windows [k, WIN] float32, start sample indices)."""
    if mode in ("train_peak", "train_zscore"):
        if mode == "train_peak":
            # Training notebook: peak-normalise the clip to -3 dBFS, then crop/zero-pad
            # the END to 64000 samples. No mean/variance normalisation.
            peak = float(np.max(np.abs(x16)))
            x = x16 * ((10 ** (PEAK_TARGET_DBFS / 20)) / (peak + 1e-6))
        else:
            # Same windows; z-score over the analysed clip (before padding).
            x = (x16 - x16.mean()) / np.sqrt(x16.var() + 1e-5)
        starts = window_starts(len(x))
        wins = []
        for s in starts:
            w = x[s : s + WIN]
            if len(w) < WIN:
                w = np.pad(w, (0, WIN - len(w)))
            wins.append(w)
        return np.stack(wins).astype(np.float32), starts
    if mode == "api_replica":
        # Deployed upstream API (src/api/inference.py): first 48000 samples,
        # zero-pad to 48000, z-score, then zero-pad to 64000 in predict().
        # Kept only as a comparison; it differs from training.
        w = x16[:48000]
        w = np.pad(w, (0, 48000 - len(w)))
        w = (w - w.mean()) / np.sqrt(w.var() + 1e-5)
        w = np.pad(w, (0, WIN - len(w)))
        return w[None, :].astype(np.float32), [0]
    raise ValueError(f"unknown preprocessing mode {mode!r}; use {PREPROCESSING_MODES}")


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


# ---- model ------------------------------------------------------------------------
def load_session(model_path: str, verify_sha: bool = True):
    import onnxruntime as ort

    if verify_sha:
        got = sha256_file(model_path)
        if got != MODEL_SHA256:
            raise RuntimeError(f"model sha256 mismatch: {got} != pinned {MODEL_SHA256}")
    so = ort.SessionOptions()
    sess = ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess


def describe_session(sess) -> dict:
    import onnxruntime as ort

    return {
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "session_providers": sess.get_providers(),
        "inputs": [(i.name, i.shape, i.type) for i in sess.get_inputs()],
        "outputs": [(o.name, o.shape, o.type) for o in sess.get_outputs()],
    }


def interface_self_test(sess, seed: int = 0) -> dict:
    """Probe what the checkpoint actually accepts (metadata alone proves nothing)."""
    rng = np.random.default_rng(seed)
    name = sess.get_inputs()[0].name
    out = {}
    base = (rng.standard_normal(WIN) * 0.1).astype(np.float32)
    ref = sess.run(None, {name: base[None]})[0]
    out["len_64000_batch1"] = {"ok": True, "shape": list(ref.shape)}
    for n in (32000, 48000, 96000):
        try:
            r = sess.run(None, {name: base[:n][None] if n <= WIN else np.tile(base, 2)[:n][None]})[0]
            out[f"len_{n}_batch1"] = {"ok": True, "shape": list(r.shape)}
        except Exception as e:
            out[f"len_{n}_batch1"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    try:
        b = np.stack([base, base[::-1].copy()])
        r = sess.run(None, {name: b})[0]
        r1 = sess.run(None, {name: base[::-1].copy()[None]})[0]
        # Colab run 20261024T102407Z: batch of 2 returned shape [1, 2] (a row is
        # silently dropped), so "ok" requires one output row per input row.
        out["batch2"] = {
            "ok": r.shape[0] == 2,
            "shape": list(r.shape),
            "max_abs_diff_vs_individual": float(
                max(np.abs(r[0] - ref[0]).max(), np.abs(r[1] - r1[0]).max())
            ) if r.shape[0] == 2 else None,
        }
    except Exception as e:
        out["batch2"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    # determinism
    again = sess.run(None, {name: base[None]})[0]
    out["deterministic"] = bool(np.allclose(again, ref))
    return out


@dataclass
class Result:
    file: str
    status: str = "ok"
    reject_reason: str = ""
    file_sha256: str = ""
    native_sr: int | None = None
    channels: int | None = None
    info: dict = field(default_factory=dict)
    preprocessing: str = ""
    window_starts_s: list = field(default_factory=list)
    window_logits: list = field(default_factory=list)
    window_scores: list = field(default_factory=list)
    clip_score_mean: float | None = None
    clip_score_max: float | None = None
    predicted_label: str = ""
    latency_ms_model: float | None = None
    latency_ms_total: float | None = None


def analyse(sess, path: str, mode: str = "train_peak") -> Result:
    """Score an audio file (any format load_audio can decode)."""
    t0 = time.perf_counter()
    res = Result(file=path, preprocessing=mode)
    try:
        res.file_sha256 = sha256_file(path)
        y, sr = load_audio(path)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, sr, t0)


def analyse_waveform(sess, samples: np.ndarray, sr: int, mode: str = "train_peak",
                     label: str = "<waveform>") -> Result:
    """Score in-memory PCM ([n] or [channels, n], float in [-1, 1]).

    Same validation, preprocessing and windowing as analyse(); intended for
    audio windows that never touch disk (e.g. remote-call PCM later).
    """
    t0 = time.perf_counter()
    res = Result(file=label, preprocessing=mode)
    y = np.atleast_2d(np.asarray(samples, dtype=np.float32))
    if y.size == 0:
        res.status, res.reject_reason = "rejected", "empty audio (0 samples)"
    elif not np.all(np.isfinite(y)):
        res.status, res.reject_reason = "rejected", "audio contains NaN/Inf samples"
    if res.status == "rejected":
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, int(sr), t0)


def _score(sess, res: Result, y: np.ndarray, sr: int, t0: float) -> Result:
    try:
        res.native_sr, res.channels = sr, int(y.shape[0])
        x16 = to_mono_16k(y, sr)
        x16, info = validate(x16)
        res.info = info
        wins, starts = make_windows(x16, res.preprocessing)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res

    name = sess.get_inputs()[0].name
    logits = []
    tm = time.perf_counter()
    for w in wins:  # batch=1 only: batch>1 drops rows (see interface_self_test)
        logits.append(sess.run(None, {name: w[None]})[0][0])
    res.latency_ms_model = round((time.perf_counter() - tm) * 1000, 1)
    logits = np.asarray(logits, dtype=np.float64)
    scores = softmax(logits)[:, SYNTHETIC_INDEX]
    res.window_starts_s = [round(s / SR, 3) for s in starts]
    res.window_logits = [[round(float(a), 4) for a in row] for row in logits]
    res.window_scores = [round(float(s), 4) for s in scores]
    res.clip_score_mean = round(float(scores.mean()), 4)
    res.clip_score_max = round(float(scores.max()), 4)
    res.predicted_label = LABELS[SYNTHETIC_INDEX] if res.clip_score_mean >= DECISION_THRESHOLD else LABELS[0]
    res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
    return res


def result_row(res: Result, **extra) -> dict:
    d = asdict(res)
    info = d.pop("info")
    for k in ("window_starts_s", "window_logits", "window_scores"):
        d[k] = json.dumps(d[k])
    row = {**extra, **d, **info}
    row.update(
        decision_threshold=DECISION_THRESHOLD,
        label_mapping=json.dumps(LABELS),
        model_repo=MODEL_REPO,
        model_revision=MODEL_REVISION,
        model_sha256=MODEL_SHA256,
    )
    return row

In [ ]:
%%writefile eval_lib.py
"""Helpers for the Dhwani generalisation evaluation (evaluation only, no training).

Standard library only, so the selection, archive access and metric code can be
unit-tested without ML dependencies (evaluation/tests/test_eval_lib.py).
Model execution, preprocessing and windowing are NOT reimplemented here: the
notebook calls src/vcd/dhwani_infer.py (embedded verbatim, read-only).

Labels follow dhwani_infer.LABELS: "genuine" (0) / "synthetic" (1).
Scores are uncalibrated synthetic-class softmax means, not probabilities.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict

# ---- pinned dataset sources (verified 2026-10-24, see evaluation/README.md) --------
SOURCES = {
    "asvspoof2019_la_eval": {
        "title": "ASVspoof 2019 Logical Access (LA), evaluation partition",
        "landing_page": "https://datashare.ed.ac.uk/handle/10283/3336",
        "doi": "10.7488/ds/2555",
        "archive": "LA.zip",
        "url": "https://datashare.ed.ac.uk/server/api/core/bitstreams/"
               "a9f87c35-f055-4015-80e2-2fdff0d46269/content",
        "bytes": 7640952520,
        "md5": "30c98f11d8b2bc21f2c257bfd78bb5c5",  # DSpace checkSum (MD5) == HTTP ETag
        "last_modified": "Tue, 04 Jun 2019 09:38:20 GMT",
        "license": "ODC-By 1.0 (archive README.txt, COPYING section)",
        "protocol_suffix": "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
        "audio_dir_suffix": "ASVspoof2019_LA_eval/flac/",
        "audio_ext": ".flac",
        # Literature values, checked against the protocol in the notebook (warn on mismatch).
        "expected_counts": {"genuine": 7355, "synthetic": 63882},
        "expected_attacks": [f"A{i:02d}" for i in range(7, 20)],
    },
    "in_the_wild": {
        "title": "In-the-Wild audio deepfake dataset (Mueller et al., Interspeech 2022)",
        "landing_page": "https://huggingface.co/datasets/mueller91/In-The-Wild",
        "project_page": "https://deepfake-total.com/in_the_wild",
        "hf_repo": "mueller91/In-The-Wild",
        "revision": "eee168f92c367f8c82ff2cf42b6f61e362fd6211",
        "archive": "release_in_the_wild.zip",
        "url": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/"
               "eee168f92c367f8c82ff2cf42b6f61e362fd6211/release_in_the_wild.zip",
        "bytes": 8161489023,
        "sha256": "46665f30a6758b45642c659c7e0e80da2ba3d121b064716824a56a010ddf9e1a",  # LFS oid
        "xet_hash": "ec530d2076f973d40a8e388e559888f1b4fea467db8126c811d34dd508ee106f",  # CDN ETag
        "license": "CONFLICT: HF card says CC-BY-SA-4.0; Fraunhofer project page says Apache-2.0",
        "meta_suffix": "meta.csv",
        "expected_counts": {"genuine": 19963, "synthetic": 11816},
    },
}

GENUINE, SYNTHETIC = "genuine", "synthetic"


# ---- remote zip access via HTTP range requests -------------------------------------
class RangeUnsupported(RuntimeError):
    """Server ignored the Range header (would have streamed the whole archive)."""


class HTTPRangeFile(io.RawIOBase):
    """Read-only, seekable view of a remote file using HTTP Range requests.

    zipfile.ZipFile can open it directly, so only the central directory and the
    selected members are transferred. zipfile checks each member's CRC-32 on read.
    `fetch(start, end_inclusive) -> bytes` can be injected for tests.
    """

    def __init__(self, url: str | None = None, size: int | None = None, fetch=None,
                 timeout: float = 60.0, retries: int = 4):
        super().__init__()
        self.url, self.timeout, self.retries = url, timeout, retries
        self._final_url = None
        self._fetch = fetch or self._http_fetch
        self.headers = {}
        if size is None:
            size = self._probe_size()
        self.size, self.pos = int(size), 0
        self.n_requests, self.bytes_fetched = 0, 0

    def _resolve(self):
        req = urllib.request.Request(self.url, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            if r.status != 206:
                raise RangeUnsupported(f"HTTP {r.status} to a Range request: {self.url}")
            self._final_url = r.geturl()
            self.headers = {k.lower(): v for k, v in r.headers.items()}
            r.read()
        return self.headers

    def _probe_size(self) -> int:
        h = self._resolve()
        return int(h["content-range"].rsplit("/", 1)[1])

    def _http_fetch(self, start: int, end: int) -> bytes:
        last = None
        for attempt in range(self.retries):
            if self._final_url is None:
                self._resolve()
            req = urllib.request.Request(self._final_url, headers={"Range": f"bytes={start}-{end}"})
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeUnsupported(f"HTTP {r.status} to a Range request")
                    data = r.read()
                if len(data) != end - start + 1:
                    raise IOError(f"short read {len(data)} != {end - start + 1}")
                return data
            except RangeUnsupported:
                raise
            except (urllib.error.URLError, IOError, TimeoutError) as e:
                last = e
                self._final_url = None  # signed redirect URLs expire: resolve again
                time.sleep(2 ** attempt)
        raise IOError(f"range fetch {start}-{end} failed after {self.retries} attempts: {last}")

    # RawIOBase interface
    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self.pos = offset
        elif whence == io.SEEK_CUR:
            self.pos += offset
        elif whence == io.SEEK_END:
            self.pos = self.size + offset
        else:
            raise ValueError(whence)
        if self.pos < 0:
            raise ValueError("negative seek position")
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        data = self._fetch(self.pos, self.pos + n - 1)
        b[: len(data)] = data
        self.pos += len(data)
        self.n_requests += 1
        self.bytes_fetched += len(data)
        return len(data)


def open_remote_zip(url: str, expected_bytes: int | None = None, buffer_size: int = 1 << 18):
    """Return (ZipFile, raw HTTPRangeFile) for a remote archive."""
    import zipfile

    raw = HTTPRangeFile(url)
    if expected_bytes is not None and raw.size != expected_bytes:
        raise IOError(f"remote size {raw.size} != recorded {expected_bytes}")
    zf = zipfile.ZipFile(io.BufferedReader(raw, buffer_size=buffer_size))
    return zf, raw


def find_member(names, suffix: str) -> str:
    hits = [n for n in names if n.endswith(suffix) and not n.split("/")[-1].startswith("._")]
    if len(hits) != 1:
        raise KeyError(f"expected exactly one member ending in {suffix!r}, found {hits[:5]}")
    return hits[0]


def file_digest(path: str, algo: str = "sha256", chunk: int = 1 << 22) -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- metadata parsing ------------------------------------------------------------
def parse_asvspoof_cm_protocol(text: str) -> list[dict]:
    """ASVspoof 2019 LA CM protocol: SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY."""
    out = []
    for i, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"protocol line {i}: expected 5 columns, got {len(parts)}: {line!r}")
        spk, utt, _unused, system, key = parts
        if key not in ("bonafide", "spoof"):
            raise ValueError(f"protocol line {i}: unknown key {key!r}")
        if key == "spoof" and system == "-":
            raise ValueError(f"protocol line {i}: spoof without attack ID")
        out.append({
            "clip_id": utt, "speaker_id": spk,
            "attack_id": system if key == "spoof" else "",
            "label": SYNTHETIC if key == "spoof" else GENUINE,
            "original_label": key,
        })
    return out


ITW_LABELS = {"bona-fide": GENUINE, "bonafide": GENUINE, "spoof": SYNTHETIC}


def parse_itw_meta(text: str) -> list[dict]:
    """In-the-Wild meta.csv (expected columns: file, speaker, label)."""
    rows = list(csv.DictReader(io.StringIO(text)))
    if not rows:
        raise ValueError("meta.csv is empty")
    cols = {c.strip().lower(): c for c in rows[0].keys()}
    missing = {"file", "speaker", "label"} - set(cols)
    if missing:
        raise ValueError(f"meta.csv lacks columns {missing}; has {list(rows[0].keys())}")
    out = []
    for i, r in enumerate(rows, 2):
        lab = r[cols["label"]].strip()
        if lab.lower() not in ITW_LABELS:
            raise ValueError(f"meta.csv line {i}: unknown label {lab!r}")
        fn = r[cols["file"]].strip()
        out.append({
            "clip_id": fn, "speaker_id": r[cols["speaker"]].strip(), "attack_id": "",
            "label": ITW_LABELS[lab.lower()], "original_label": lab,
        })
    return out


def describe_records(records: list[dict]) -> dict:
    by_label = Counter(r["label"] for r in records)
    attacks = Counter(r["attack_id"] for r in records if r["label"] == SYNTHETIC and r["attack_id"])
    spk = defaultdict(Counter)
    for r in records:
        spk[r["label"]][r["speaker_id"]] += 1
    return {
        "n": len(records),
        "by_label": dict(sorted(by_label.items())),
        "by_attack": dict(sorted(attacks.items())),
        "n_speakers_by_label": {k: len(v) for k, v in sorted(spk.items())},
        "n_speakers_total": len({r["speaker_id"] for r in records}),
        "duplicate_clip_ids": len(records) - len({r["clip_id"] for r in records}),
    }


# ---- score-free, reproducible selection ------------------------------------------
FORBIDDEN_SELECTION_KEYS = ("score", "logit", "pred", "prob")


def _rng(seed: int, *stratum) -> random.Random:
    # str seeds are hashed with SHA-512 by random.seed (version 2): stable across
    # runs, machines and PYTHONHASHSEED. One RNG per stratum => order-independent.
    return random.Random(":".join([str(seed), *map(str, stratum)]))


def speaker_round_robin(records: list[dict], n: int, seed: int, *stratum) -> list[dict]:
    """Take n clips spreading over speakers: shuffled speaker order, shuffled clips per
    speaker, then one clip per speaker per round. Uses metadata only."""
    by_spk = defaultdict(list)
    for r in sorted(records, key=lambda r: r["clip_id"]):
        by_spk[r["speaker_id"]].append(r)
    spks = sorted(by_spk)
    _rng(seed, *stratum, "speakers").shuffle(spks)
    for s in spks:
        _rng(seed, *stratum, "clips", s).shuffle(by_spk[s])
    out, rnd = [], 0
    while len(out) < n and any(len(by_spk[s]) > rnd for s in spks):
        for s in spks:
            if len(out) >= n:
                break
            if len(by_spk[s]) > rnd:
                out.append(by_spk[s][rnd])
        rnd += 1
    return out


def select_subset(records: list[dict], dataset: str, seed: int, n_genuine: int,
                  n_synthetic: int | None = None, n_per_attack: int | None = None,
                  full: bool = False, exclude_clip_ids=(), exclude_speakers=()) -> tuple[list[dict], dict]:
    """Score-free stratified sample. full=True: every record (sorted by ID).

    exclude_clip_ids / exclude_speakers: clips already used as development
    evidence (e.g. an earlier pilot) are removed BEFORE sampling, so a fresh
    held-out subset never shares clips (or speakers) with them.
    """
    for r in records[:1]:
        bad = [k for k in r if any(f in k.lower() for f in FORBIDDEN_SELECTION_KEYS)]
        assert not bad, f"selection input contains detector-derived fields: {bad}"
    ex_ids, ex_spk = set(exclude_clip_ids), set(exclude_speakers)
    n_before = len(records)
    records = [r for r in records if r["clip_id"] not in ex_ids and r["speaker_id"] not in ex_spk]
    exclusion = {"excluded_clip_ids": len(ex_ids), "excluded_speakers": len(ex_spk),
                 "records_before": n_before, "records_after": len(records)}
    gen = [r for r in records if r["label"] == GENUINE]
    syn = [r for r in records if r["label"] == SYNTHETIC]
    if full:
        sel = [dict(r, stratum="all") for r in sorted(records, key=lambda r: r["clip_id"])]
        rule = "full evaluation partition: every protocol/metadata row, sorted by clip_id"
    else:
        sel = [dict(r, stratum="genuine") for r in speaker_round_robin(gen, n_genuine, seed, dataset, GENUINE)]
        if n_per_attack is not None:
            by_atk = defaultdict(list)
            for r in syn:
                by_atk[r["attack_id"]].append(r)
            for atk in sorted(by_atk):
                take = speaker_round_robin(by_atk[atk], n_per_attack, seed, dataset, SYNTHETIC, atk)
                sel += [dict(r, stratum=f"synthetic/{atk}") for r in take]
            rule = (f"genuine: {n_genuine} by speaker round-robin; synthetic: {n_per_attack} per attack ID "
                    f"({len(by_atk)} attacks) by speaker round-robin within attack")
        else:
            sel += [dict(r, stratum="synthetic") for r in speaker_round_robin(syn, n_synthetic, seed, dataset, SYNTHETIC)]
            rule = f"genuine: {n_genuine}, synthetic: {n_synthetic}, each by speaker round-robin"
        rule += (f"; seed {seed}; per-stratum random.Random('{seed}:{dataset}:<label>[:<attack>]:...'); "
                 "input sorted by clip_id; metadata only (no audio, no detector output)")
    for i, r in enumerate(sel):
        r["selection_rank"] = i
        r["dataset"] = dataset
    ids = "\n".join(r["clip_id"] for r in sel)
    meta = {
        "dataset": dataset, "seed": seed, "full": full, "rule": rule,
        "n_selected": len(sel), "selected": describe_records(sel), "available": describe_records(records),
        "clip_id_list_sha256": hashlib.sha256(ids.encode()).hexdigest(),
        "shortfalls": {}, "exclusion": exclusion,
    }
    assert not ({r["clip_id"] for r in sel} & ex_ids) and not ({r["speaker_id"] for r in sel} & ex_spk)
    if not full:
        got = Counter(r["stratum"] for r in sel)
        if got["genuine"] < n_genuine:
            meta["shortfalls"]["genuine"] = f"{got['genuine']} < {n_genuine}"
        for k, v in got.items():
            if k.startswith("synthetic/") and n_per_attack and v < n_per_attack:
                meta["shortfalls"][k] = f"{v} < {n_per_attack}"
    return sel, meta


# ---- metrics (positive class = synthetic; higher score = more synthetic) --------
def _split(labels, scores):
    pos = [s for l, s in zip(labels, scores) if l == 1]
    neg = [s for l, s in zip(labels, scores) if l == 0]
    return pos, neg


def roc_auc(labels, scores):
    """Mann-Whitney U / (n_pos * n_neg), ties counted 1/2. None if a class is empty."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None
    allv = sorted([(s, 1) for s in pos] + [(s, 0) for s in neg])
    rank_sum_pos, i = 0.0, 0
    while i < len(allv):
        j = i
        while j < len(allv) and allv[j][0] == allv[i][0]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0  # ranks i+1..j
        rank_sum_pos += avg_rank * sum(1 for k in range(i, j) if allv[k][1] == 1)
        i = j
    u = rank_sum_pos - len(pos) * (len(pos) + 1) / 2.0
    return u / (len(pos) * len(neg))


def eer(labels, scores):
    """EER with the ASVspoof 2019 reference convention (compute_eer in the official
    tDCF package): sweep thresholds over all scores, take the point minimising
    |FNR - FPR| and report their mean. Decision: synthetic if score >= t.
    Returns (eer, threshold_at_eer) or (None, None). The EER threshold is
    descriptive only; it is NOT an operating point for this project."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None, None
    pts = sorted(set(pos) | set(neg)) + [math.inf]  # inf: FNR = 1, FPR = 0
    pos_s, neg_s = sorted(pos), sorted(neg)
    best = None
    ip = ineg = 0
    for t in pts:  # FNR(t) = #pos < t / P ; FPR(t) = #neg >= t / N
        while ip < len(pos_s) and pos_s[ip] < t:
            ip += 1
        while ineg < len(neg_s) and neg_s[ineg] < t:
            ineg += 1
        fnr, fpr = ip / len(pos_s), (len(neg_s) - ineg) / len(neg_s)
        d = abs(fnr - fpr)
        if best is None or d < best[0]:
            best = (d, (fnr + fpr) / 2.0, t)
    return best[1], best[2]


def wilson(k: int, n: int, z: float = 1.959964):
    if n == 0:
        return None, None
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def at_threshold(labels, scores, thr: float) -> dict:
    tp = sum(1 for l, s in zip(labels, scores) if l == 1 and s >= thr)
    fn = sum(1 for l, s in zip(labels, scores) if l == 1 and s < thr)
    fp = sum(1 for l, s in zip(labels, scores) if l == 0 and s >= thr)
    tn = sum(1 for l, s in zip(labels, scores) if l == 0 and s < thr)
    P, N = tp + fn, fp + tn
    rec = tp / P if P else None
    fpr = fp / N if N else None
    spec = tn / N if N else None
    return {
        "threshold": thr,
        "confusion": {"tp_synthetic_as_synthetic": tp, "fn_synthetic_as_genuine": fn,
                      "fp_genuine_as_synthetic": fp, "tn_genuine_as_genuine": tn},
        "synthetic_recall": rec, "synthetic_recall_wilson95": wilson(tp, P),
        "genuine_fpr": fpr, "genuine_fpr_wilson95": wilson(fp, N),
        "genuine_specificity": spec,
        "balanced_accuracy": (rec + spec) / 2 if rec is not None and spec is not None else None,
        "accuracy": (tp + tn) / (P + N) if P + N else None,
        "n_synthetic": P, "n_genuine": N,
    }


def bootstrap_ci(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """Stratified (within-class) percentile bootstrap, 95 %. Reflects clip sampling
    only; it ignores speaker clustering, so intervals are optimistic."""
    pos, neg = _split(labels, scores)
    if not pos or not neg or n_boot <= 0:
        return {}
    rng = random.Random(f"bootstrap:{seed}")
    acc = defaultdict(list)
    for _ in range(n_boot):
        p = [pos[rng.randrange(len(pos))] for _ in pos]
        q = [neg[rng.randrange(len(neg))] for _ in neg]
        lab, sc = [1] * len(p) + [0] * len(q), p + q
        acc["roc_auc"].append(roc_auc(lab, sc))
        acc["eer"].append(eer(lab, sc)[0])
        acc["balanced_accuracy@thr"].append(at_threshold(lab, sc, thr)["balanced_accuracy"])
    out = {}
    for k, v in acc.items():
        v = sorted(v)
        out[k] = (v[int(0.025 * (len(v) - 1))], v[int(math.ceil(0.975 * (len(v) - 1)))])
    out["n_boot"], out["seed"] = n_boot, seed
    return out


def evaluate(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """labels: 1 = synthetic, 0 = genuine (scored clips only)."""
    e, et = eer(labels, scores)
    return {
        "n_scored": len(labels),
        "threshold_free": {"roc_auc": roc_auc(labels, scores), "eer": e,
                           "eer_threshold_descriptive_only": et},
        "at_threshold": at_threshold(labels, scores, thr),
        "bootstrap95": bootstrap_ci(labels, scores, thr, n_boot, seed),
    }


REJECT_CATEGORIES = (
    ("too short", "too_short"), ("mostly silent", "mostly_silent"), ("silent", "silent"),
    ("unreadable", "unreadable"), ("empty", "empty"), ("NaN", "non_finite"),
)


def reject_category(reason: str) -> str:
    for needle, cat in REJECT_CATEGORIES:
        if reason.startswith(needle) or needle in reason[:40]:
            return cat
    return "other"


def jdump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

In [ ]:
%%writefile antideepfake_hf.py
"""Candidate detector: AntiDeepfake Wav2Vec2-Small (NII Yamagishi Lab), loaded WITHOUT fairseq.

Model: nii-yamagishilab/wav2vec-small-anti-deepfake @ MODEL_REVISION, model.safetensors
(380,210,632 B, sha256 MODEL_SHA256). Licence CC BY-NC-SA 4.0 (research/educational use;
NOT for commercial deployment without permission). Base: facebook/wav2vec2-base (Apache-2.0).
Paper: Ge, Wang, Liu, Yamagishi, "Post-training for Deepfake Speech Detection", arXiv:2506.21090.
We did not train or fine-tune this model.

Why this loader exists: the published inference code builds the encoder with
fairseq 0.12.2 (Python <= 3.8 wheels; the project is archived), which does not install on
current Colab runtimes. The checkpoint's encoder keys are fairseq wav2vec2 names, so we map
them onto transformers' Wav2Vec2Model using the same correspondence as HF's official
convert_wav2vec2_original_pytorch_checkpoint_to_pytorch.py, and FAIL LOUDLY if any
parameter is left over or missing. Numerical identity with the fairseq path is NOT
proven here (see verify_notes()); the notebook checks behaviour on development evidence
before any held-out scoring.

Architecture correspondence, checked against fairseq v0.12.2 fairseq/models/wav2vec/wav2vec2.py and
transformers modeling_wav2vec2.py (code read side by side; see architecture_check()):
  * conv feature extractor: 7 Conv1d(no bias) [(512,10,5),(512,3,2)x4,(512,2,2)x2]; fairseq
    extractor_mode='default' = GroupNorm(512,512) after conv 0 only, GELU after every conv
    == HF feat_extract_norm='group', conv_bias=False, feat_extract_activation='gelu'.
  * LayerNorm(512) on frames, then Linear(512,768) (fairseq model.layer_norm + post_extract_proj
    == HF feature_projection.layer_norm + .projection).
  * positional conv: Conv1d(768,768,k=128,pad=64,groups=16), weight_norm(dim=2), drop last frame
    (SamePad == Wav2Vec2SamePadLayer), GELU; x = x + pos.
  * post-LN transformer (fairseq layer_norm_first=False == HF do_stable_layer_norm=False):
    encoder.layer_norm BEFORE the 12 layers; each layer: attn -> residual -> layer_norm -> FFN(GELU,
    3072) -> residual -> final_layer_norm; 12 heads, scaling head_dim**-0.5; exact (erf) GELU both.
    No final norm after the last layer in either implementation. Output = last_hidden_state == fairseq 'x'.
  * fairseq pads frames to a multiple of 2 with a key-padding mask and removes the pad afterwards;
    with the pad masked this is a no-op for the result. HF does no such padding. No attention mask is
    used by either side for a single unpadded clip (batch size 1).
  * head: mean over time (AdaptiveAvgPool1d(1)) -> Linear(768,2) -> softmax; fake = index 0.
  * eval mode: no time/feature masking, no dropout, no layerdrop on either side.
  Differences that remain: the authors decode with torchaudio and resample with torchaudio.functional.resample
  (our path: soundfile/librosa + soxr; irrelevant at 16 kHz input), and they score whole files while we apply
  the 2-10 s policy (first 10 s). Numerical parity is only established by the optional fairseq parity cell.

Published recipe (model card), reproduced here:
  mono (mean) -> 16 kHz -> layer_norm over the whole waveform (zero mean, unit var, eps 1e-5)
  -> encoder last hidden state -> mean over time -> Linear(768, 2) -> softmax.
  Output order: index 0 = FAKE, index 1 = REAL (card: "<Fake score, Real score>").
  Arbitrary input length (no cropping in the published code).
"""

from __future__ import annotations

import time

try:
    import numpy as np
except ImportError:  # the key-mapping functions are stdlib-only and testable without numpy
    np = None

MODEL_REPO = "nii-yamagishilab/wav2vec-small-anti-deepfake"
MODEL_REVISION = "9a13264b5dcc827a8d5a4f8e01fccefa392f886b"
MODEL_FILE = "model.safetensors"
MODEL_SHA256 = "828ee456122f86d5d631cb7895a10e5c62c78a4fcb8a8b1c42cb5838a9abcfe0"
MODEL_BYTES = 380210632
LICENSE = "CC-BY-NC-SA-4.0 (weights); base facebook/wav2vec2-base Apache-2.0; code BSD-3"
FAKE_INDEX = 0          # softmax index of the synthetic class (card: <Fake, Real>)
# The card reports EER 4.24 % "@ 0.8128" on In-the-Wild. That threshold is the EER point found ON In-the-Wild
# with the authors' whole-file pipeline, i.e. it was selected on the dataset we test on. It is therefore NOT a
# pre-registered operating point for us and is reported only as context, never as a decision threshold.
CARD_EER_THRESHOLD_ITW_CONTEXT_ONLY = 0.8128
SR = 16000

# Keys of the fairseq checkpoint that belong to pre-training only (quantiser, projections,
# mask embedding). They have no counterpart in inference and are dropped on purpose.
PRETRAIN_ONLY_PREFIXES = ("quantizer.", "project_q.", "final_proj.", "mask_emb")


def _map_encoder_key(k: str) -> str | None:
    """fairseq Wav2Vec2Model key (prefix 'm_ssl.model.' removed) -> transformers Wav2Vec2Model key."""
    if k.startswith(PRETRAIN_ONLY_PREFIXES):
        return None
    if k.startswith("feature_extractor.conv_layers."):
        _, _, layer, typ, param = k.split(".")
        if typ == "0":
            return f"feature_extractor.conv_layers.{layer}.conv.{param}"
        if typ == "2":  # group norm (layer 0 only, feat_extract_norm='group')
            return f"feature_extractor.conv_layers.{layer}.layer_norm.{param}"
        raise KeyError(k)
    if k.startswith("post_extract_proj."):
        return "feature_projection.projection." + k.split(".", 1)[1]
    if k.startswith("layer_norm."):
        return "feature_projection.layer_norm." + k.split(".", 1)[1]
    if k.startswith("encoder.pos_conv.0."):
        param = k.rsplit(".", 1)[1]           # weight_g / weight_v / bias
        return "encoder.pos_conv_embed.conv." + param
    if k.startswith("encoder.layer_norm."):
        return k
    if k.startswith("encoder.layers."):
        _, _, i, rest = k.split(".", 3)
        sub = {
            "self_attn.k_proj": "attention.k_proj", "self_attn.v_proj": "attention.v_proj",
            "self_attn.q_proj": "attention.q_proj", "self_attn.out_proj": "attention.out_proj",
            "self_attn_layer_norm": "layer_norm", "fc1": "feed_forward.intermediate_dense",
            "fc2": "feed_forward.output_dense", "final_layer_norm": "final_layer_norm",
        }
        for src, dst in sub.items():
            if rest.startswith(src + "."):
                return f"encoder.layers.{i}.{dst}.{rest[len(src) + 1:]}"
        raise KeyError(k)
    raise KeyError(k)


def convert_state_dict(sd: dict, hf_keys: set) -> tuple[dict, dict, dict]:
    """Return (encoder_state_dict, head_state_dict, report). Raises on anything unexplained."""
    enc, head, dropped, unexplained = {}, {}, [], []
    for k, v in sd.items():
        if k.startswith("proj_fc."):
            head[k.split(".", 1)[1]] = v
            continue
        if not k.startswith("m_ssl.model."):
            unexplained.append(k)
            continue
        kk = k[len("m_ssl.model."):]
        try:
            mk = _map_encoder_key(kk)
        except KeyError:
            unexplained.append(k)
            continue
        if mk is None:
            dropped.append(k)
            continue
        # weight-norm naming differs across torch/transformers versions
        if mk.endswith(("weight_g", "weight_v")) and mk not in hf_keys:
            alt = mk.replace("weight_g", "parametrizations.weight.original0").replace(
                "weight_v", "parametrizations.weight.original1")
            if alt in hf_keys:
                mk = alt
        enc[mk] = v
    missing = sorted(hf_keys - set(enc))
    unexpected = sorted(set(enc) - hf_keys)
    report = {"n_checkpoint_tensors": len(sd), "n_encoder_mapped": len(enc), "n_head": len(head),
              "dropped_pretrain_only": dropped, "unexplained": unexplained,
              "missing_in_checkpoint": missing, "unexpected_for_hf": unexpected}
    if unexplained or missing or unexpected or set(head) != {"weight", "bias"}:
        raise RuntimeError(f"checkpoint/model mismatch: {report}")
    return enc, head, report


# fairseq Wav2Vec2Config defaults that the authors' code relies on (only 5 fields are set explicitly there)
EXPECTED_HF_CONFIG = {
    "model_type": "wav2vec2", "feat_extract_norm": "group", "conv_bias": False, "feat_extract_activation": "gelu",
    "conv_dim": [512] * 7, "conv_kernel": [10, 3, 3, 3, 3, 2, 2], "conv_stride": [5, 2, 2, 2, 2, 2, 2],
    "hidden_size": 768, "num_hidden_layers": 12, "num_attention_heads": 12, "intermediate_size": 3072,
    "hidden_act": "gelu", "do_stable_layer_norm": False, "num_conv_pos_embeddings": 128,
    "num_conv_pos_embedding_groups": 16, "layer_norm_eps": 1e-05, "add_adapter": False,
}


def architecture_check(cfg) -> dict:
    """Compare the HF config against the fairseq-default architecture the checkpoint was trained with."""
    got = {k: getattr(cfg, k) for k in EXPECTED_HF_CONFIG}
    bad = {k: (got[k], v) for k, v in EXPECTED_HF_CONFIG.items() if got[k] != v}
    if bad:
        raise RuntimeError(f"HF config does not match the fairseq architecture: {bad}")
    return got


class AntiDeepfakeHF:
    """Thin inference wrapper (torch + transformers). Build with .load()."""

    def __init__(self, encoder, head, device, report):
        self.encoder, self.head, self.device, self.report = encoder, head, device, report

    @classmethod
    def load(cls, safetensors_path: str, config_source: str, device: str = "cpu"):
        import torch
        from safetensors.torch import load_file
        from transformers import Wav2Vec2Config, Wav2Vec2Model

        cfg = Wav2Vec2Config.from_pretrained(config_source)
        arch = architecture_check(cfg)
        # inference only: no masking, no layerdrop, no dropout
        cfg.mask_time_prob = 0.0
        cfg.mask_feature_prob = 0.0
        cfg.layerdrop = 0.0
        cfg.apply_spec_augment = False
        encoder = Wav2Vec2Model(cfg)
        sd = load_file(safetensors_path)
        hf_keys = set(encoder.state_dict().keys()) - {"masked_spec_embed"}
        enc_sd, head_sd, report = convert_state_dict(sd, hf_keys)
        res = encoder.load_state_dict(enc_sd, strict=False)
        assert not res.unexpected_keys and set(res.missing_keys) <= {"masked_spec_embed"}, res
        head = torch.nn.Linear(768, 2)
        head.load_state_dict(head_sd)
        encoder.eval().to(device)
        head.eval().to(device)
        report["architecture"] = arch
        report["n_params_encoder"] = sum(p.numel() for p in encoder.parameters())
        report["n_params_head"] = sum(p.numel() for p in head.parameters())
        return cls(encoder, head, device, report)

    @staticmethod
    def preprocess(x16: np.ndarray) -> np.ndarray:
        """Published recipe: layer_norm over the whole waveform (eps 1e-5)."""
        x = x16.astype(np.float64)
        return ((x - x.mean()) / np.sqrt(x.var() + 1e-5)).astype(np.float32)

    def logits(self, wave: np.ndarray) -> np.ndarray:
        """wave: preprocessed mono 16 kHz float32 [n]. Returns logits [2] (index 0 = fake)."""
        import torch

        with torch.no_grad():
            t = torch.from_numpy(wave)[None].to(self.device)
            h = self.encoder(t).last_hidden_state          # [1, T, 768]
            z = self.head(h.mean(dim=1))                    # [1, 2]
        return z[0].float().cpu().numpy()


def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def score_clip(model: AntiDeepfakeHF, x16: np.ndarray, window=None, hop=None) -> dict:
    """Primary: whole analysed clip in one pass (as published). If window/hop are given,
    additionally scores 4 s windows (mean of fake scores) for comparability with Dhwani."""
    t0 = time.perf_counter()
    x = model.preprocess(x16)
    z = model.logits(x)
    p = softmax(z)
    out = {"logits_fake_real": [round(float(a), 4) for a in z], "fake_score_whole": round(float(p[FAKE_INDEX]), 6),
           "n_samples": int(len(x16))}
    if window:
        n = len(x)
        starts = [0] if n <= window else list(range(0, n - window + 1, hop))
        if n > window and starts[-1] + window < n:
            starts.append(n - window)
        ws = []
        for s in starts:
            w = x[s:s + window]
            if len(w) < window:
                w = np.pad(w, (0, window - len(w)))
            ws.append(float(softmax(model.logits(w))[FAKE_INDEX]))
        out.update(window_starts_s=[round(s / SR, 3) for s in starts], window_fake_scores=[round(v, 6) for v in ws],
                   fake_score_window_mean=round(float(np.mean(ws)), 6))
    out["latency_ms_model"] = round((time.perf_counter() - t0) * 1000, 1)
    return out


def verify_notes() -> str:
    return ("Weights were loaded through a fairseq->transformers key mapping (every checkpoint tensor accounted "
            "for) and the architecture fields were checked against fairseq defaults. This does NOT prove numerical "
            "parity with the authors' fairseq code; parity is 'unverified' unless the fairseq parity cell ran "
            "and reported a small max difference. The development-evidence gate (pilot clips) is a stop-and-"
            "investigate check: a low AUC does not by itself prove the mapping is wrong, and a high AUC does not "
            "prove parity.")

In [ ]:
%%writefile fairseq_parity_ref.py
"""Reference-implementation scorer for the parity check. Runs in a SEPARATE Python 3.8 venv with
fairseq==0.12.2 (the authors' dependency), never in the main Colab kernel.

Usage: python fairseq_parity_ref.py <model.safetensors> <inputs.npz> <out.npz>
inputs.npz: arrays w0, w1, ... = already layer-normed mono 16 kHz float32 waveforms (identical to
what the transformers path receives). Writes logits (fake, real) per input, computed exactly like the
model card's code: fairseq Wav2Vec2Model(features_only=True)['x'] -> mean over time -> proj_fc.
"""
import sys

import numpy as np
import torch
from safetensors.torch import load_file
from fairseq.models.wav2vec import Wav2Vec2Model, Wav2Vec2Config

model_path, in_path, out_path = sys.argv[1:4]
cfg = Wav2Vec2Config(encoder_layers=12, encoder_embed_dim=768, quantize_targets=True, latent_dim=256, final_dim=256)
ssl = Wav2Vec2Model(cfg)
sd = load_file(model_path)
enc = {k[len("m_ssl.model."):]: v for k, v in sd.items() if k.startswith("m_ssl.model.")}
missing, unexpected = ssl.load_state_dict(enc, strict=False)
assert not unexpected, unexpected
assert not missing, missing   # the checkpoint carries every fairseq tensor, incl. quantizer
head = torch.nn.Linear(768, 2)
head.load_state_dict({k.split(".", 1)[1]: v for k, v in sd.items() if k.startswith("proj_fc.")})
ssl.eval(); head.eval()
inp = np.load(in_path)
out = {}
with torch.no_grad():
    for k in sorted(inp.files):
        x = torch.from_numpy(inp[k].astype(np.float32))[None]
        feats = ssl(x, mask=False, features_only=True)["x"]        # [1, T, 768]
        out[k] = head(feats.mean(dim=1))[0].numpy()
np.savez(out_path, **out)
print("fairseq", getattr(__import__("fairseq"), "__version__", "?"), "torch", torch.__version__, "scored", len(out))

In [ ]:
%%writefile itw_pilot_clip_ids.txt
10055.wav
8846.wav
26460.wav
13957.wav
3520.wav
13659.wav
4914.wav
20156.wav
20520.wav
5705.wav
10864.wav
13586.wav
5020.wav
777.wav
30117.wav
2906.wav
28252.wav
13583.wav
15467.wav
5784.wav
19894.wav
27151.wav
30441.wav
5967.wav
6075.wav
17368.wav
4590.wav
234.wav
27706.wav
12681.wav
21309.wav
17921.wav
30610.wav
30665.wav
5052.wav
14782.wav
25073.wav
5511.wav
8618.wav
19890.wav
3236.wav
21906.wav
1194.wav
27536.wav
24162.wav
9266.wav
22081.wav
2803.wav
2513.wav
11044.wav
3910.wav
15829.wav
4199.wav
20859.wav
26603.wav
9027.wav
20495.wav
1645.wav
18847.wav
25114.wav
1558.wav
23318.wav
22423.wav
2726.wav
3708.wav
23195.wav
13421.wav
1959.wav
11729.wav
16428.wav
7852.wav
27564.wav
22745.wav
21486.wav
30389.wav
10756.wav
26404.wav
5268.wav
21273.wav
16049.wav
11124.wav
28491.wav
5937.wav
18833.wav
24648.wav
27937.wav
1323.wav
29120.wav
16744.wav
28925.wav
5319.wav
9107.wav
4541.wav
20952.wav
1542.wav
26962.wav
6971.wav
1159.wav
10476.wav
5792.wav
29606.wav
2165.wav
10400.wav
9548.wav
31078.wav
10516.wav
26179.wav
29927.wav
19088.wav
28222.wav
2923.wav
12185.wav
26297.wav
29785.wav
24562.wav
11225.wav
16164.wav
15660.wav
4605.wav
21464.wav
12720.wav
22217.wav
27252.wav
29835.wav
30559.wav
18831.wav
9278.wav
14378.wav
16528.wav
23722.wav
13152.wav
5490.wav
29059.wav
6154.wav
21050.wav
1575.wav
12368.wav
9924.wav
28015.wav
29657.wav
703.wav
15148.wav
7293.wav
21238.wav
22274.wav
7818.wav
25771.wav
6606.wav
31752.wav
28742.wav
19041.wav
10241.wav
25765.wav
14840.wav
10496.wav
20736.wav
11835.wav
31118.wav
9851.wav
27941.wav
16855.wav
25438.wav
24262.wav
19715.wav
29129.wav
16034.wav
15487.wav
14411.wav
8837.wav
6718.wav
31176.wav
3936.wav
15865.wav
22330.wav
22036.wav
21372.wav
13027.wav
10902.wav
5387.wav
15381.wav
1428.wav
25229.wav
30695.wav
14579.wav
4266.wav
12142.wav
27970.wav
17174.wav
1309.wav
15860.wav
2817.wav
20336.wav
344.wav
16256.wav
30066.wav
4084.wav
1845.wav
4129.wav
25957.wav
25610.wav

In [ ]:
# 3. Imports, code provenance, environment
import os, json, time, hashlib, platform, datetime, importlib, traceback, shutil
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.getcwd())
import dhwani_infer as dh, eval_lib as el, antideepfake_hf as ad
for m in (dh, el, ad): importlib.reload(m)
EXPECTED = {
    "dhwani_infer.py": "8a7c9c8f4275fb27a0d6ba3dde43aaeedf98974b7057378c37122467ba756077",
    "eval_lib.py": "1b61f04ac27d3734c22d905cc5fc8bfcb48fac708e5233459ac5c6c1ff2ffca0",
    "antideepfake_hf.py": "fba4d117443e6ff10dd0b7fdd64f89cff94960a73b8f0a85799ec884eb2b3db5",
    "fairseq_parity_ref.py": "b329d63588642ec781cb309f0f46c3661795cc3a262d865e7c2acc5da707692c"
}
CODE_SHA256 = {f: hashlib.sha256((open(f, encoding="utf-8").read().rstrip() + "\n").encode()).hexdigest() for f in EXPECTED}
PARITY = {"status": "unverified", "attempted": False}   # overwritten by cell 13 only if the reference run succeeds
assert CODE_SHA256 == EXPECTED, CODE_SHA256
PILOT_IDS = [l.strip() for l in open("itw_pilot_clip_ids.txt") if l.strip()]
assert len(PILOT_IDS) == 200 and len(set(PILOT_IDS)) == 200
assert dh.DECISION_THRESHOLD == 0.5 and dh.LABELS == {0: "genuine", 1: "synthetic"}
import onnxruntime as ort, transformers, librosa, soundfile, sklearn
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
           onnxruntime=ort.__version__, numpy=np.__version__, pandas=pd.__version__, librosa=librosa.__version__,
           soundfile=soundfile.__version__, sklearn=sklearn.__version__, device=DEVICE,
           gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None, code_sha256=CODE_SHA256)
print(json.dumps(ENV, indent=1))
if DEVICE != "cuda": print("WARNING: no GPU; the candidate will run on CPU (slower, still fine for 500 clips)")

In [ ]:
# 4. Checkpoints: Dhwani (pinned, sha256) and candidate (pinned revision, sha256). Cached copies reused.
from huggingface_hub import hf_hub_download
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); os.makedirs(DRIVE_DIR, exist_ok=True)
def get(repo, fname, rev, sha, nbytes):
    cands = [os.path.join(DRIVE_DIR, fname)] if USE_DRIVE else []
    try: cands.append(hf_hub_download(repo, fname, revision=rev, local_files_only=True))
    except Exception: pass
    p = next((c for c in cands if os.path.exists(c) and os.path.getsize(c) == nbytes), None)
    src = "cache"
    if p is None:
        src = "downloaded"; print("downloading", repo, fname, f"({nbytes/1e9:.2f} GB)")
        p = hf_hub_download(repo, fname, revision=rev, **({"local_dir": DRIVE_DIR} if USE_DRIVE else {}))
    assert os.path.getsize(p) == nbytes
    got = dh.sha256_file(p); assert got == sha, f"{fname} sha256 mismatch: {got}"
    print(f"{src}: {p} sha256 ok"); return p, src
DH_PATH, DH_SRC = get(dh.MODEL_REPO, dh.MODEL_FILE, dh.MODEL_REVISION, dh.MODEL_SHA256, dh.MODEL_BYTES)
AD_PATH, AD_SRC = get(ad.MODEL_REPO, ad.MODEL_FILE, ad.MODEL_REVISION, ad.MODEL_SHA256, ad.MODEL_BYTES)
AD_CFG = hf_hub_download(ad.MODEL_REPO, "config.json", revision=ad.MODEL_REVISION)

In [ ]:
# 5. Load both models. Candidate: strict fairseq->transformers mapping (fails loudly on any leftover tensor)
sess = dh.load_session(DH_PATH, verify_sha=False)
PROBE = dh.interface_self_test(sess)
assert PROBE["len_64000_batch1"]["ok"]
cand = ad.AntiDeepfakeHF.load(AD_PATH, os.path.dirname(AD_CFG), device=DEVICE)
print(json.dumps({k: v for k, v in cand.report.items() if k not in ("dropped_pretrain_only", "architecture")}, indent=1))
print("architecture fields == fairseq defaults:", cand.report["architecture"])
print("dropped (pre-training only):", cand.report["dropped_pretrain_only"])
assert cand.report["n_params_encoder"] + cand.report["n_params_head"] > 94_000_000
print(ad.verify_notes())

In [ ]:
# 6. Run folder
BASE = os.path.join(DRIVE_DIR if USE_DRIVE else WORK, "runs")
utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_NAME = RUN_NAME or f"{utc}_heldout_itw_dhwani_vs_antideepfake_small"
RUN_DIR = os.path.join(BASE, RUN_NAME); os.makedirs(RUN_DIR, exist_ok=True)
for d in ("audio/dev", "audio/heldout"): os.makedirs(os.path.join(WORK, d), exist_ok=True)
SETTINGS = dict(HELDOUT_SEED=HELDOUT_SEED, HELDOUT_N=HELDOUT_N, MIN_PILOT_DEV_AUC=MIN_PILOT_DEV_AUC, CAND_WINDOW=CAND_WINDOW,
                CAND_HOP=CAND_HOP, EXPORT_ONNX=EXPORT_ONNX, dhwani=dict(repo=dh.MODEL_REPO, revision=dh.MODEL_REVISION,
                sha256=dh.MODEL_SHA256, preprocessing="train_peak", clip_score="mean of 4 s windows", threshold=0.5),
                candidate=dict(repo=ad.MODEL_REPO, revision=ad.MODEL_REVISION, sha256=ad.MODEL_SHA256, license=ad.LICENSE,
                recipe="layer_norm whole clip, mean-pool, fake=softmax[0]", thresholds=[0.5],
                card_eer_threshold_context_only=ad.CARD_EER_THRESHOLD_ITW_CONTEXT_ONLY),
                policy=dict(min_s=dh.MIN_DUR_S, max_s=dh.MAX_DUR_S), batch_size=1)
el.jdump({"settings": SETTINGS, "env": ENV, "probe": PROBE, "candidate_load_report": cand.report, "started_utc": utc},
         os.path.join(RUN_DIR, "run_config.json"))
print(RUN_DIR)

In [ ]:
# 7. In-the-Wild: open archive (HTTP range), parse meta.csv, build DEV (pilot 200) and HELD-OUT (fresh, clip-disjoint) manifests
src = el.SOURCES["in_the_wild"]
zf, raw = el.open_remote_zip(src["url"], src["bytes"])
names = zf.namelist(); meta_m = el.find_member(names, src["meta_suffix"]); prefix = meta_m[:-len("meta.csv")]
recs = el.parse_itw_meta(zf.read(meta_m).decode("utf-8"))
nameset = set(names)
for r in recs: r["archive_member"] = prefix + r["clip_id"] if prefix + r["clip_id"] in nameset else ""
desc = el.describe_records(recs); print("meta.csv:", desc)
assert desc["by_label"] == src["expected_counts"], desc["by_label"]

pilot_set = set(PILOT_IDS)
DEV = [dict(r, dataset="in_the_wild", split="dev_pilot") for r in recs if r["clip_id"] in pilot_set]
assert len(DEV) == 200
HELD, hmeta = el.select_subset(recs, "in_the_wild", HELDOUT_SEED, HELDOUT_N, n_synthetic=HELDOUT_N, exclude_clip_ids=pilot_set)
for r in HELD: r["split"] = "heldout"
assert not ({r["clip_id"] for r in HELD} & pilot_set)
hmeta["speaker_overlap_with_pilot"] = len({r["speaker_id"] for r in HELD} & {r["speaker_id"] for r in DEV})
hmeta["note"] = "clip-disjoint from pilot; speaker-disjoint impossible (pilot covered all 54 speakers)"
el.jdump(hmeta, os.path.join(RUN_DIR, "selection_heldout_in_the_wild.json"))
pd.DataFrame(HELD).to_csv(os.path.join(RUN_DIR, "manifest_heldout_in_the_wild.csv"), index=False)
pd.DataFrame(DEV).to_csv(os.path.join(RUN_DIR, "manifest_dev_pilot_in_the_wild.csv"), index=False)
print("held-out:", hmeta["n_selected"], hmeta["selected"]["by_label"], "| speakers:", hmeta["selected"]["n_speakers_by_label"],
      "| clip-ID sha256:", hmeta["clip_id_list_sha256"], "| shortfalls:", hmeta["shortfalls"] or "none")
print("speakers shared with pilot:", hmeta["speaker_overlap_with_pilot"], "(expected: most; see note)")

In [ ]:
# 8. Scoring helpers (both models on the same decoded, validated audio; batch size 1)
def fetch(m, split):
    local = os.path.join(WORK, "audio", split, os.path.basename(m["archive_member"]))
    if not os.path.exists(local):
        data = zf.read(m["archive_member"])          # CRC-32 checked by zipfile
        with open(local, "wb") as f: f.write(data)
    return local

def score_both(m, split, run_dhwani=True):
    row = {k: m.get(k, "") for k in ("dataset", "split", "clip_id", "label", "speaker_id", "selection_rank")}
    t0 = time.perf_counter()
    try:
        local = fetch(m, split)
        row["file_sha256"] = dh.sha256_file(local)
        y, sr = dh.load_audio(local); row.update(native_sr=sr, channels=int(y.shape[0]))
        x16 = dh.to_mono_16k(y, sr); x16, info = dh.validate(x16); row.update(info); row["status"] = "ok"
    except dh.AudioRejected as e:
        row.update(status="rejected", reject_reason=str(e), reject_category=el.reject_category(str(e)))
        row["latency_ms_total"] = round((time.perf_counter() - t0) * 1000, 1); return row
    except Exception as e:
        row.update(status="fetch_failed", reject_reason=f"{type(e).__name__}: {e}", reject_category="fetch_failed")
        row["latency_ms_total"] = round((time.perf_counter() - t0) * 1000, 1); return row
    # candidate (published recipe + secondary window variant)
    c = ad.score_clip(cand, x16, window=CAND_WINDOW, hop=CAND_HOP)
    row.update(cand_fake_score=c["fake_score_whole"], cand_logits_fake_real=json.dumps(c["logits_fake_real"]),
               cand_fake_score_window_mean=c["fake_score_window_mean"], cand_window_scores=json.dumps(c["window_fake_scores"]),
               cand_latency_ms=c["latency_ms_model"])
    if run_dhwani:
        r = dh.analyse(sess, local, mode="train_peak")   # re-decodes; identical validation path
        assert r.status == "ok", r.reject_reason
        row.update(dhwani_score=r.clip_score_mean, dhwani_score_max=r.clip_score_max, dhwani_window_scores=json.dumps(r.window_scores),
                   dhwani_window_logits=json.dumps(r.window_logits), dhwani_latency_ms=r.latency_ms_model)
    row["latency_ms_total"] = round((time.perf_counter() - t0) * 1000, 1)
    return row

def run_split(manifest, split, run_dhwani, path):
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)["clip_id"] for l in open(path) if l.strip()}
    todo = [m for m in manifest if m["clip_id"] not in done]
    print(f"[{split}] {len(manifest)} clips, {len(done)} done, {len(todo)} to go")
    t0 = time.time()
    with open(path, "a", encoding="utf-8") as out:
        for i, m in enumerate(todo):
            out.write(json.dumps(score_both(m, split, run_dhwani), default=str) + "\n"); out.flush()
            if (i + 1) % 25 == 0 or i + 1 == len(todo):
                e = time.time() - t0; print(f"  {i+1}/{len(todo)} {e/60:.1f} min, ETA {e/(i+1)*(len(todo)-i-1)/60:.1f} min")
    return pd.DataFrame([json.loads(l) for l in open(path) if l.strip()]).drop_duplicates("clip_id", keep="last")

In [ ]:
# 9. DEV-EVIDENCE GATE: candidate on the 200 pilot clips (Dhwani already scored there; not repeated).
#    AUC < MIN_PILOT_DEV_AUC -> stop and investigate. This does NOT establish that the mapping is wrong, and a high
#    AUC does NOT establish parity (cell 13 does that). Nothing is tuned or changed based on this value.
DEV_PATH = os.path.join(RUN_DIR, "predictions_dev_pilot.jsonl")
D = run_split(DEV, "dev", run_dhwani=False, path=DEV_PATH)
ok = D[D.status == "ok"]
y = (ok.label == "synthetic").astype(int).tolist()
dev_auc = el.roc_auc(y, ok.cand_fake_score.tolist()); dev_eer = el.eer(y, ok.cand_fake_score.tolist())[0]
print(f"candidate on dev pilot: n={len(ok)}  AUC {dev_auc:.4f}  EER {dev_eer:.4f}  (published full-ITW: AUC 0.993, EER 0.042)")
print("Dhwani on the same clips (run 20261024T123319Z, 155 scored): AUC 0.456, EER 0.561")
GATE = {"dev_n_scored": int(len(ok)), "candidate_dev_auc": dev_auc, "candidate_dev_eer": dev_eer, "min_auc": MIN_PILOT_DEV_AUC,
        "passed": dev_auc is not None and dev_auc >= MIN_PILOT_DEV_AUC,
        "meaning": "stop-and-investigate gate on development evidence; neither a proof of a mapping error nor of parity"}
el.jdump(GATE, os.path.join(RUN_DIR, "dev_evidence_gate.json"))
assert GATE["passed"], ("STOP: candidate AUC on development clips is below the gate. Investigate (run cell 13 parity first; check "
                        "decoding/normalisation) before any held-out scoring. Report the value as is.")

In [ ]:
# 10. HELD-OUT scoring: both models, same clips (resumable)
HO_PATH = os.path.join(RUN_DIR, "predictions_heldout.jsonl")
H = run_split(HELD, "heldout", run_dhwani=True, path=HO_PATH)
assert len(H) == len(HELD)

In [ ]:
# 11. Coverage (separate from accuracy) and metrics: threshold-free first, then fixed thresholds; paired bootstrap
cov = H.groupby("label").agg(selected=("clip_id", "size"), scored=("status", lambda s: int((s == "ok").sum())),
                             rejected=("status", lambda s: int((s == "rejected").sum())),
                             fetch_failed=("status", lambda s: int((s == "fetch_failed").sum())))
cov["coverage"] = (cov.scored / cov.selected).round(4)
print("HELD-OUT coverage:\n", cov.to_string()); print("\nrejections:", H[H.status != "ok"].reject_category.value_counts().to_dict() or "none")
OK = H[H.status == "ok"].copy(); yy = (OK.label == "synthetic").astype(int).tolist()
SC = {"dhwani_train_peak_mean4s": OK.dhwani_score.astype(float).tolist(),
      "candidate_whole_clip": OK.cand_fake_score.astype(float).tolist(),
      "candidate_window_mean4s": OK.cand_fake_score_window_mean.astype(float).tolist()}
THR = {"dhwani_train_peak_mean4s": [0.5], "candidate_whole_clip": [0.5], "candidate_window_mean4s": [0.5]}
METRICS = {}
for k, s in SC.items():
    m = el.evaluate(yy, s, THR[k][0], n_boot=1000, seed=HELDOUT_SEED)
    m["at_thresholds"] = {str(t): el.at_threshold(yy, s, t) for t in THR[k]}
    METRICS[k] = m
    tf, bs = m["threshold_free"], m["bootstrap95"]
    print(f"\n== {k}  (n={len(yy)}; HELD-OUT PILOT SUBSET, clip-disjoint from dev, speakers shared)")
    print(f"   ROC-AUC {tf['roc_auc']:.4f} [{bs['roc_auc'][0]:.3f}, {bs['roc_auc'][1]:.3f}]   EER {tf['eer']:.4f} [{bs['eer'][0]:.3f}, {bs['eer'][1]:.3f}]")
    for t, a in m["at_thresholds"].items():
        c = a["confusion"]
        print(f"   @ {t}: TP {c['tp_synthetic_as_synthetic']} FN {c['fn_synthetic_as_genuine']} FP {c['fp_genuine_as_synthetic']} TN {c['tn_genuine_as_genuine']} | "
              f"bal.acc {a['balanced_accuracy']:.3f}  recall {a['synthetic_recall']:.3f}  FPR {a['genuine_fpr']:.3f}")
# paired bootstrap: AUC(candidate) - AUC(dhwani) on the same clips
import random
rng = random.Random(f"paired:{HELDOUT_SEED}"); pos = [i for i, v in enumerate(yy) if v == 1]; neg = [i for i, v in enumerate(yy) if v == 0]
diffs = []
for _ in range(1000):
    idx = [pos[rng.randrange(len(pos))] for _ in pos] + [neg[rng.randrange(len(neg))] for _ in neg]
    l = [yy[i] for i in idx]
    diffs.append(el.roc_auc(l, [SC["candidate_whole_clip"][i] for i in idx]) - el.roc_auc(l, [SC["dhwani_train_peak_mean4s"][i] for i in idx]))
diffs.sort(); PAIRED = {"auc_diff_point": METRICS["candidate_whole_clip"]["threshold_free"]["roc_auc"] - METRICS["dhwani_train_peak_mean4s"]["threshold_free"]["roc_auc"],
                        "auc_diff_95": [diffs[25], diffs[974]]}
print("\nPaired AUC difference (candidate - Dhwani):", PAIRED)
print(f"Threshold 0.5 fixed in advance for both models; nothing was tuned here. The card's EER threshold {ad.CARD_EER_THRESHOLD_ITW_CONTEXT_ONLY} "
      "was found on In-the-Wild itself and is NOT applied (context only). EER thresholds printed by eval_lib are descriptive.")

In [ ]:
# 12. Deployment feasibility: ONNX export of the candidate + ORT parity (NOT a mobile build; evidence for the Android path)
ONNX_REPORT = {"attempted": EXPORT_ONNX}
if EXPORT_ONNX:
    class Wrapped(torch.nn.Module):
        def __init__(s, enc, head): super().__init__(); s.enc, s.head = enc, head
        def forward(s, wav): return s.head(s.enc(wav).last_hidden_state.mean(dim=1))
    w = Wrapped(cand.encoder, cand.head).eval().cpu()
    onnx_path = os.path.join(WORK, "antideepfake_small.onnx")
    try:
        dummy = torch.randn(1, 64000)
        torch.onnx.export(w, dummy, onnx_path, input_names=["input"], output_names=["logits"], opset_version=17,
                          dynamic_axes={"input": {1: "time"}})
        s2 = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        diffs, lat = [], []
        for m in OK.head(8).itertuples():
            x16 = dh.to_mono_16k(*dh.load_audio(os.path.join(WORK, "audio", "heldout", os.path.basename(m.clip_id))))
            x16, _ = dh.validate(x16); x = ad.AntiDeepfakeHF.preprocess(x16)
            t0 = time.perf_counter(); zo = s2.run(None, {"input": x[None]})[0][0]; lat.append((time.perf_counter() - t0) * 1000)
            zt = w(torch.from_numpy(x)[None]).detach().numpy()[0]
            diffs.append(float(np.abs(ad.softmax(zo)[0] - ad.softmax(zt)[0])))
        ONNX_REPORT.update(ok=True, file_mb=round(os.path.getsize(onnx_path) / 1e6, 1), opset=17,
                           max_abs_fake_score_diff=max(diffs), ort_cpu_ms_median=float(np.median(lat)), n_clips=len(diffs),
                           note="fp32 ONNX, dynamic time axis, Colab CPU latency; int8/fp16 and on-device latency not measured")
    except Exception as e:
        ONNX_REPORT.update(ok=False, error=f"{type(e).__name__}: {str(e)[:500]}")
    cand.encoder.to(DEVICE); cand.head.to(DEVICE)
print(json.dumps(ONNX_REPORT, indent=1))

In [ ]:
# 13. PARITY with the authors' reference implementation: separate Python 3.8 venv + fairseq 0.12.2, identical inputs.
#     Outcome is recorded as verified (with max |diff|) or unverified (with the reason). Never silently skipped.
PARITY = {"status": "unverified", "attempted": TRY_FAIRSEQ_PARITY}
if TRY_FAIRSEQ_PARITY:
    VENV = "/content/fsenv"; PY = f"{VENV}/bin/python"
    t0 = time.time()
    try:
        steps = [["uv", "python", "install", "3.8"], ["uv", "venv", "--python", "3.8", VENV],
                 ["uv", "pip", "install", "--python", PY, "torch==2.4.1", "torchaudio==2.4.1", "--index-url", "https://download.pytorch.org/whl/cpu"],
                 ["uv", "pip", "install", "--python", PY, "fairseq==0.12.2", "safetensors==0.5.3", "numpy<1.25"]]
        for cmd in steps:
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"{' '.join(cmd[:3])} failed: {(r.stderr or r.stdout)[-600:]}")
        PARITY["env_setup_s"] = round(time.time() - t0, 1)
        # identical inputs: the exact arrays the transformers path scores (layer-normed, 2-10 s policy applied)
        sample = OK.head(PARITY_N_CLIPS)
        arrays, hf_logits = {}, {}
        for n, m in enumerate(sample.itertuples()):
            x16 = dh.to_mono_16k(*dh.load_audio(os.path.join(WORK, "audio", "heldout", os.path.basename(m.clip_id))))
            x16, _ = dh.validate(x16); x = ad.AntiDeepfakeHF.preprocess(x16)
            arrays[f"w{n:03d}"] = x; hf_logits[f"w{n:03d}"] = cand.logits(x)
        np.savez(os.path.join(WORK, "parity_inputs.npz"), **arrays)
        r = subprocess.run([PY, "fairseq_parity_ref.py", AD_PATH, os.path.join(WORK, "parity_inputs.npz"), os.path.join(WORK, "parity_ref.npz")],
                           capture_output=True, text=True, timeout=1800)
        if r.returncode != 0:
            raise RuntimeError("reference run failed: " + (r.stderr or r.stdout)[-800:])
        ref = np.load(os.path.join(WORK, "parity_ref.npz"))
        dl = [float(np.max(np.abs(ref[k] - hf_logits[k]))) for k in arrays]
        dp = [float(abs(ad.softmax(ref[k])[0] - ad.softmax(hf_logits[k])[0])) for k in arrays]
        PARITY.update(status="verified" if max(dp) < 1e-3 else "MISMATCH", n_clips=len(dl), max_abs_logit_diff=max(dl),
                      max_abs_fake_score_diff=max(dp), reference_stdout=r.stdout.strip()[-300:], total_s=round(time.time() - t0, 1),
                      note="fairseq 0.12.2 (authors' dependency) on CPU vs transformers mapping on " + DEVICE)
    except Exception as e:
        PARITY.update(status="unverified", reason=f"{type(e).__name__}: {str(e)[:800]}", total_s=round(time.time() - t0, 1))
print(json.dumps(PARITY, indent=1))
if PARITY["status"] != "verified":
    print("\nCONVERSION PARITY UNVERIFIED (or mismatched): report the candidate numbers with this caveat.")

In [ ]:
# 14. Save everything (no audio) and download
H.to_csv(os.path.join(RUN_DIR, "predictions_heldout.csv"), index=False)
D.to_csv(os.path.join(RUN_DIR, "predictions_dev_pilot_candidate.csv"), index=False)
cov.reset_index().to_csv(os.path.join(RUN_DIR, "coverage_heldout.csv"), index=False)
RUNTIME = {"candidate_ms_median": float(OK.cand_latency_ms.median()), "dhwani_ms_median": float(OK.dhwani_latency_ms.median()),
           "device_candidate": DEVICE, "provider_dhwani": sess.get_providers()[0]}
el.jdump({"run_name": RUN_NAME, "scope": "HELD-OUT PILOT SUBSET of In-the-Wild (clip-disjoint from dev pilot; speakers shared). Not a full-dataset benchmark.",
          "settings": SETTINGS, "env": ENV, "conversion_gate": GATE, "heldout_selection": hmeta, "coverage": cov.reset_index().to_dict("records"),
          "metrics": METRICS, "paired_auc_difference": PAIRED, "onnx": ONNX_REPORT, "conversion_parity": PARITY, "runtime": RUNTIME,
          "candidate_license": ad.LICENSE, "overlap_notes": "In-the-Wild not in candidate training table; VoxCeleb2 (celebrity speech) is -> speaker overlap possible. Dhwani overlap unknown.",
          "finished_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")}, os.path.join(RUN_DIR, "metrics.json"))
zip_path = shutil.make_archive(os.path.join(WORK, RUN_NAME), "zip", RUN_DIR)
print(sorted(os.listdir(RUN_DIR)), "\n", zip_path, f"{os.path.getsize(zip_path)/1e6:.2f} MB")
try:
    from google.colab import files; files.download(zip_path)
except ImportError: pass

## Report back
The run zip (no audio) and the executed notebook, into `evaluation/runs/<run_name>/`. Paste the outputs of cells 5, 7, 9, 11, 12 and 13.

## How to read the result
- Cell 9 is a stop-and-investigate gate on development evidence, not a result and not proof about the conversion either way.
- Cell 11 is the result: a held-out pilot subset of In-the-Wild (300 clips planned, minus < 2 s rejections). Threshold-free numbers first;
  0.5 fixed before scoring. The clips share speakers with the pilot (unavoidable), so generalisation to new speakers is untested.
- Cell 13 decides whether "conversion parity" is **verified** (max fake-score difference vs the authors' fairseq code) or **unverified**.
  Every candidate number must carry that status.
- Any further model, recipe or threshold decision taken after seeing this result again needs a fresh selection (new seed, exclude both the pilot and these clip IDs).